In [ ]:
import os
from dotenv import load_dotenv

from langchain_ollama import OllamaEmbeddings

load_dotenv(override=True)

OLLAMA_BASE_URL: str = os.getenv("OLLAMA_BASE_URL", "")
EMBEDDING_MODEL: str = os.getenv("EMBEDDING_MODEL", "")

embedding = OllamaEmbeddings(base_url=OLLAMA_BASE_URL, model=EMBEDDING_MODEL)

In [ ]:
from langchain_classic.document_loaders import CSVLoader
from langchain_text_splitters import CharacterTextSplitter

loader = CSVLoader("./survey.csv")
docs = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents=docs)

In [ ]:
from langchain_classic.vectorstores import Chroma

db = Chroma.from_documents(documents=docs, embedding=embedding)

In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

OPENAI_BASE_URL: str = os.getenv("OPENAI_BASE_URL", "")
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL: str = os.getenv("OPENAI_MODEL_NAME", "")

llm = ChatOpenAI(
    base_url=OPENAI_BASE_URL,
    api_key=SecretStr(OPENAI_API_KEY),
    model=LLM_MODEL,
    temperature=0,
    max_completion_tokens=250,
    reasoning_effort="none",
)

In [ ]:
from langchain_core.prompts.chat import PromptTemplate  # type: ignore

template = """You are a clothing assistant chatbot.

Answer the customer's questions only using the source data provided. Please answer their specific question.
If you are unsure, say \"I don't know, please call our customer support\".
Keep your answer concise.

{context}
"""

prompt = PromptTemplate(template=template, input_variables=["context"])

formatted_prompt = prompt.format(
    context="A customer is on the clothing company website and wants to chat with the website chatbot. They will ask you a question, please answer to their specific question."
)

In [ ]:
from langchain_classic.chains import RetrievalQA

chain_type_kwargs = {"prompt": prompt}

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 1}),
    chain_type_kwargs=chain_type_kwargs,
)

In [ ]:
# query = "What is the exchange policy if the size was not fit for me?"
# query = "I want to buy one to gift, are you offering wrapper or something else?"

query = "Hey, I am checking on something, which was there before 2 days, now i am not finding. How can i check if they are in stocks? "
response = chain.invoke(query)
print(response["result"])